# Lab 19: Tree-Based Models — Random Forests
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnostic Lab | 65 min Core + 55 min AI + 15 min Extension

---

**Learning Objectives:**
- Find the bug in a model comparison that makes one model look better than it is, and measure how much better it really is
- Explain why a feature-importance ranking cannot support a policy recommendation, and why impurity (MDI) and permutation importance rank the same features differently
- Tune a forest by cross-validation, and compare it with Ridge and XGBoost on the test rows
- Read SHAP values for a boosted model: how far each feature moves each prediction, and in which direction
- Write your own prompt for a function that summarises SHAP values, and check every number it gives against the lab's

**Dataset:** California Housing (scikit-learn): 20,640 block groups, 8 features, and the median house value in units of $100,000. You have used it since Topic 4.

**Format:** Parts 1 and 2 each show you an analyst's code and conclusion, and ask what is wrong with them; Part 3 asks you to compare models and read what drives one of them. Write your answers in the text cell under each set of questions.

**Deliverable:** your fix in Part 1, your written answers in Parts 1-3, and the AI-Assisted Expansion (your prompts, the checking log and three answers).

**Time:** Core (65 min) + AI-Assisted Expansion (required, 55 min) + Extension (optional, 15 min)

With setup, the README and the GitHub submission, plan on about 2 hours 15 minutes, most of it after class. The optional Extension adds about 15 minutes.

**AI in the manual parts:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion while you work on those parts. AI-assisted coding is allowed only in the AI-Assisted Expansion, where you write the prompt yourself.

---

> ### What you write, and what you run
>
> **You write code:**
> - **Part 1:** the corrected comparison, with the forest's corrected R² beside the analyst's. About four lines in an empty cell.
> - **Extension (optional):** three waterfall charts, from a short outline.
>
> **You write, in text cells:**
> - two answers in Part 1, three in Part 2 and two in Part 3;
> - **the AI-Assisted Expansion (required):** your own prompt, from scratch, asking AI for a function that summarises a model's SHAP values as Part 3 does; a revised prompt for round 2 if round 1 shows a `NO`, a `missing` or an error (and round 3 if you need it); a short checking log; and three answers. Your prompts, the log and the answers are what is graded, not the AI's code, which you paste rather than type.
>
> **You run and read:** everything else. That is the analyst's two cells, permutation importance, the tuning search, XGBoost, the four-model table, and XGBoost's SHAP values, summary and chart. Read each cell before you run it: the questions ask about what it prints, and the AI section asks for Part 3's summary again. The check cells in the AI section: run them; you need not follow their code.

## Setup

The first line of the setup cell installs the versions of three libraries this notebook was
measured with: scikit-learn 1.9, XGBoost 3.4 and SHAP 0.52. Each changes the numbers or the
errors you see: scikit-learn builds trees differently before 1.9 (the tree and forest numbers
differ in the third or fourth decimal), XGBoost before 3.3 builds a different boosted model, and
SHAP's versions differ in which XGBoost models they can read and what they do with them. The cell
prints the three versions. If they are not 1.9.x, 3.4.x and 0.52.x, use
**Runtime → Restart session** and run this cell again.

`fetch_california_housing()` downloads the data the first time (a few seconds) and returns it in
parts: `.data` holds the features, `.target` the house values and `.feature_names` the column
names. `pd.DataFrame(data.data, columns=...)` puts the features in a table with those names.
`train_test_split` keeps 20% of the rows (`test_size=0.2`) as test rows. Every model in this lab
is given `random_state=RANDOM_STATE`, so running a cell twice gives the same numbers.

In [ ]:
# GIVEN — run as-is
# Setup: the libraries, and California Housing split into training and test rows
!pip install "scikit-learn==1.9.*" "xgboost==3.4.*" "shap==0.52.*" -q

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import sklearn
import xgboost
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

RANDOM_STATE = 42          # the seed of the split and of every model
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

print(f"scikit-learn {sklearn.__version__}, xgboost {xgboost.__version__}, shap {shap.__version__}")
print(f"{len(X_train)} training rows, {len(X_test)} test rows")
print(f"features: {list(X.columns)}")

**Expected output:** versions 1.9.x, 3.4.x and 0.52.x; 16512 training rows and 4128 test
rows; and the eight features, from `MedInc` (median income, in $10,000s) to `Longitude`.

## Part 1: Find the Bug — Model Comparison (15 min)

An analyst compares three models and reports how well each predicts house values: a single
decision tree with no depth limit, Ridge with α = 1, and a random forest of 100 trees.
**Something is wrong with how the comparison is set up.** Find it, fix it, explain it.

**Reading the code.** Each model is fitted with `.fit(X, y)`, predictors first, and `.predict(X)`
gives its predictions for the rows `X`. `r2_score(y_true, y_pred)` is R², with the true values
first. `{x:.4f}` prints x with four decimals. The forest takes 10-15 seconds on free Colab.

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# Three models fitted on the training rows, and the R² the analyst reports for each
tree = DecisionTreeRegressor(random_state=RANDOM_STATE)
tree.fit(X_train, y_train)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

print("=== Model Comparison ===")
print(f"Single Tree   — R²: {r2_score(y_test, tree.predict(X_test)):.4f}")
print(f"Ridge         — R²: {r2_score(y_test, ridge.predict(X_test)):.4f}")
print(f"Random Forest — R²: {r2_score(y_train, rf.predict(X_train)):.4f}")

**Expected output:** R² of 0.6161 for the single tree, 0.5759 for Ridge and 0.9735 for the random
forest. The analyst's note under the table reads: *"Random Forest achieves R² > 0.97! Far superior
to alternatives."*

### YOUR DIAGNOSIS

1. What is wrong? Name the line, say what it does differently from the other two, and explain why that makes the forest look better than it is.
2. After your fix, below: how far apart are the forest's two R² values, and which idea from Chapter 15 does that gap measure? By how much does the forest really beat the other two?

**Your task.** In the empty cell below, print the corrected comparison, and the forest's R² from
the analyst's line beside your corrected one. Reuse the fitted `tree`, `ridge` and `rf`: nothing
needs refitting. About four lines.

**Verification checkpoint:** after the fix, the forest's R² is between 0.78 and 0.83. If you get
more than 0.95, you have not found the bug.

In [ ]:
# YOUR TASK — Part 1: the corrected comparison, and the analyst's forest R² beside it

*Your answers here:*

1.
2.

## Part 2: Find the Methodological Flaw — Feature Importance (20 min)

The analyst fits a larger forest, of 200 trees, and uses its feature importances to write a
policy memo. The code runs correctly. Read the memo and decide whether it follows.

**Reading the code.** `.feature_importances_` is the forest's impurity importance, MDI (mean
decrease in impurity). Every split lowers the squared error inside its box; MDI adds up those
drops for each feature, over every split in every tree, weighted by how many training rows reached
the split, and scales the eight totals to sum to 1. A trailing underscore marks something learned
during `.fit()`. `pd.Series(values, index=X.columns)` labels each number with its feature,
`.sort_values(ascending=False)` puts the largest first, and `.to_string()` prints the Series
without the summary line pandas adds below it. It takes 20-30 seconds on free Colab.

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# A forest of 200 trees, and its impurity-based (MDI) feature importances
rf200 = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)
rf200.fit(X_train, y_train)

importance = pd.Series(rf200.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Feature Importance (MDI):")
print(importance.round(4).to_string())

**Expected output:** MedInc 0.5258, AveOccup 0.1381, Latitude 0.0887, Longitude 0.0883, HouseAge
0.0544, AveRooms 0.0444, Population 0.0308 and AveBedrms 0.0296.

The analyst's memo reads: *"The top predictor is MedInc (importance = 0.526). Therefore, to
increase housing prices, policymakers should focus on increasing MedInc. The second most important
lever is AveOccup."*

### YOUR DIAGNOSIS

1. What is wrong with the memo's reasoning? Say what these importances measure, and what they can and cannot tell a policymaker.
2. Draw on Chapter 10's DAGs: name one thing that moves with a block group's median income and also with its house values. What would you need before claiming that raising incomes raises house values (Chapter 24)?

*Your answers here:*

1.
2.

**A second measure, on new rows.** MDI comes from the training rows. Permutation importance uses
the fitted forest and the test rows: `permutation_importance(model, X, y, n_repeats=10,
random_state=...)` shuffles one column of `X` at a time, which breaks that feature's link with the
outcome, and records how far the forest's R² falls (R² is the score it uses for a regressor unless
told otherwise). It repeats each shuffle 10 times, in different orders, seeded, and
`.importances_mean` is the average fall. `pd.DataFrame({"MDI": ..., "permutation": ...})` puts
the two Series side by side as columns, lined up by feature name. It takes a while:
80 shuffled copies of the 4,128 test rows, each predicted by 200 trees, so allow half a minute
to a minute on free Colab.

In [ ]:
# GIVEN — run as-is (half a minute to a minute)
# Permutation importance for the same forest, on the test rows, beside MDI
perm_result = permutation_importance(rf200, X_test, y_test, n_repeats=10,
                                     random_state=RANDOM_STATE)
perm_importance = pd.Series(perm_result.importances_mean, index=X.columns)

importance_table = pd.DataFrame({"MDI": importance, "permutation": perm_importance})
print(importance_table.sort_values("permutation", ascending=False).round(4).to_string())

**Expected output:**

| | MDI | permutation |
|---|---|---|
| MedInc | 0.5258 | 0.7379 |
| Latitude | 0.0887 | 0.4452 |
| Longitude | 0.0883 | 0.3382 |
| AveOccup | 0.1381 | 0.2030 |
| HouseAge | 0.0544 | 0.0728 |
| AveRooms | 0.0444 | 0.0265 |
| AveBedrms | 0.0296 | 0.0093 |
| Population | 0.0308 | 0.0086 |

### YOUR DIAGNOSIS

3. Which feature does MDI rank higher than permutation importance does, and which two does it rank lower? From how each measure is computed, give one reason MDI can credit a feature that the forest relies on less when it predicts new rows. Does the memo's "second most important lever" keep its place on the second measure?

*Your answer here:*

## Part 3: Hyperparameter Tuning + XGBoost Comparison (30 min)

Part 1's forest used scikit-learn's defaults. This Part tunes it by cross-validation, fits a
gradient-boosted model, XGBoost, and compares them on the test rows. Then it asks XGBoost which
features drive its predictions, with SHAP values.

**Tuning.** `GridSearchCV` fits a new forest for every combination in `param_grid`, on every fold,
and keeps the combination with the best average score.
- `max_depth`: 10 levels, or `None`, no limit (as in Part 1).
- `max_features`: how many of the 8 features each split may choose from. `'sqrt'` is √8, rounded
  down to 2; `0.5` is half of them, 4. Part 1's forest let every split choose from all 8.
- `n_estimators` stays at 100: more trees do not make a forest overfit (Topic 19); they only take
  longer.
- `cv=3`: three folds.
- scikit-learn always keeps the *largest* score, so to minimise the error it scores by minus the
  MSE, `'neg_mean_squared_error'`, and `np.sqrt(-score)` turns that back into an RMSE, in the
  target's units, $100,000s.
- `n_jobs=-1` uses every processor core.

That is 2 × 2 = 4 combinations × 3 folds = 12 forests, plus one to refit the winner on all the
training rows: one to two minutes on free Colab.

After the search, `.cv_results_` holds every combination's settings (`'params'`) and average score
(`'mean_test_score'`), in the same order, and `zip` walks through the two lists side by side.
`str()` turns `None` into text, and `:<5` left-aligns it in 5 characters, so the columns line up.
`.best_params_` is the winning combination, `.best_score_` its score, and `.best_estimator_` a
forest with those settings, refitted on all the training rows.

In [ ]:
# GIVEN — run as-is (one to two minutes: 13 forests)
# Tune max_depth and max_features by 3-fold cross-validation
param_grid = {
    "n_estimators": [100],
    "max_depth": [10, None],
    "max_features": ["sqrt", 0.5],
}
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=3,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

for params, score in zip(grid_search.cv_results_["params"],
                         grid_search.cv_results_["mean_test_score"]):
    print(f"max_depth = {str(params['max_depth']):<5} max_features = "
          f"{str(params['max_features']):<5} CV RMSE {np.sqrt(-score):.4f}")

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV RMSE: {np.sqrt(-grid_search.best_score_):.4f}")

best_rf = grid_search.best_estimator_
best_pred = best_rf.predict(X_test)
print(f"Tuned RF — Test RMSE: {np.sqrt(mean_squared_error(y_test, best_pred)):.4f}, "
      f"Test R²: {r2_score(y_test, best_pred):.4f}")

**Expected output:** CV RMSEs of 0.5522 (depth 10, `'sqrt'`), 0.5382 (depth 10, 0.5), 0.5066 (no
limit, `'sqrt'`) and 0.5048 (no limit, 0.5). Then `Best parameters: {'max_depth': None,
'max_features': 0.5, 'n_estimators': 100}`, `Best CV RMSE: 0.5048`, and
`Tuned RF — Test RMSE: 0.4969, Test R²: 0.8116`.

**XGBoost.** A forest grows deep trees independently and averages them. Gradient boosting grows
shallow trees one after another, each fitted to the errors the trees before it left, and adds them
up. In `XGBRegressor`, `n_estimators=200` is the number of trees, `max_depth=5` keeps each tree to
at most 2⁵ = 32 leaves, and `learning_rate=0.1` scales each tree's contribution down, so that no
single tree decides much. It uses scikit-learn's `.fit` and `.predict`, and takes a second or two.

In [ ]:
# GIVEN — run as-is
# XGBoost: 200 shallow trees, each fitted to the errors of the trees before it
xgb = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=RANDOM_STATE)
xgb.fit(X_train, y_train)

**Four models on the test rows.** The next cell computes the tuning cell's two numbers, test RMSE
and test R², for `ridge` and `rf` from Part 1 and `best_rf` and `xgb` from this Part.
`[("Ridge", ridge), ...]` is a list of (name, model) pairs, and `for name, model in ...` takes
them one pair at a time. `{name:<14}` left-aligns the name in 14 characters.

In [ ]:
# GIVEN — run as-is
# Test RMSE and test R² for four models: two from Part 1, two from this Part
for name, model in [("Ridge", ridge), ("RF (default)", rf), ("RF (tuned)", best_rf),
                    ("XGBoost", xgb)]:
    pred = model.predict(X_test)
    print(f"{name:<14}Test RMSE {np.sqrt(mean_squared_error(y_test, pred)):.4f}   "
          f"Test R² {r2_score(y_test, pred):.4f}")

**Expected output:**

| | Test RMSE | Test R² |
|---|---|---|
| Ridge | 0.7455 | 0.5759 |
| RF (default) | 0.5063 | 0.8044 |
| RF (tuned) | 0.4969 | 0.8116 |
| XGBoost | 0.4665 | 0.8339 |

**What drives XGBoost's predictions: SHAP values.** A SHAP value is one feature's share of one
prediction. For each test row, XGBoost's prediction is the *base value*, its average prediction,
plus one SHAP value for each of the eight features: a positive one pushed this row's prediction up,
a negative one pushed it down.
- `shap.TreeExplainer(model)` computes them exactly for a model made of trees. It needs no data
  besides the model: each tree recorded how many training rows went each way at every split, and
  the explainer averages over those.
- `.shap_values(X)` gives an array with one row per row of `X` and one column per feature.
  XGBoost stores the values as 32-bit numbers, good to about seven significant digits, and
  `.shape` gives the array's number of rows and columns.
- `.expected_value` is the base value. Depending on the model and the SHAP version it is a single
  number or an array holding one, so `np.ravel(...)[0]` takes its first number either way, and
  `float(...)` makes it a plain Python number.
- `a[0]` is the first row of an array, and `.sum()` adds it up; `{x:+.4f}` prints x with its sign
  and four decimals. `xgb.predict(X_test.iloc[[0]])` predicts the first test row alone: the double
  brackets keep it a one-row table.

In [ ]:
# GIVEN — run as-is
# SHAP values for XGBoost: one for each test row and feature, plus the base value
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)          # 4128 rows x 8 features
base_value = float(np.ravel(explainer.expected_value)[0])

print(f"SHAP values: {shap_values.shape[0]} rows x {shap_values.shape[1]} features")
print(f"first test row: base value {base_value:.4f} + its eight SHAP values "
      f"{shap_values[0].sum():+.4f} = {base_value + shap_values[0].sum():.4f}")
print(f"XGBoost's prediction for that row: {xgb.predict(X_test.iloc[[0]])[0]:.4f}")

**Expected output:** 4128 rows × 8 features; for the first test row, a base value of 2.0719 plus
SHAP values that add up to −1.5472 gives 0.5247, and XGBoost's prediction is 0.5247.

**A summary for each feature.** Two numbers summarise a feature's 4,128 SHAP values:
- `mean_abs_shap`, the mean of their absolute values: how far, on average, the feature moves a
  prediction, up or down. `np.abs(a).mean(axis=0)` takes it down every column at once.
- `direction`, the correlation between the feature's values and its SHAP values: near +1 when
  higher values of the feature push the prediction up, near −1 when they push it down.
  `np.corrcoef(a, b)` gives a 2 × 2 table of correlations, and `[0, 1]` is the one between a and
  b. `enumerate(X_test.columns)` gives each column's position `j` with its name `c`, and
  `[... for j, c in ...]` builds a list with one correlation per feature.

In [ ]:
# GIVEN — run as-is
# The SHAP summary: how far each feature moves a prediction on average, and in which direction
shap_table = pd.DataFrame({
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
    "direction": [np.corrcoef(X_test[c], shap_values[:, j])[0, 1]
                  for j, c in enumerate(X_test.columns)],
}, index=X_test.columns).sort_values("mean_abs_shap", ascending=False)
print(shap_table.round(4).to_string())

**Expected output:**

| | mean_abs_shap | direction |
|---|---|---|
| Latitude | 0.5241 | −0.9371 |
| Longitude | 0.4434 | −0.8938 |
| MedInc | 0.4219 | 0.9629 |
| AveOccup | 0.2020 | −0.3723 |
| AveRooms | 0.0957 | 0.5276 |
| HouseAge | 0.0554 | 0.5710 |
| AveBedrms | 0.0287 | 0.0278 |
| Population | 0.0238 | 0.3491 |

The AI-Assisted Expansion asks you to reproduce this table with a function written by AI.

**The same values as a chart.** `shap.summary_plot(shap_values, X_test)` draws a beeswarm: one
row per feature, in the order of `mean_abs_shap`, and one dot per test row, placed at its SHAP
value and coloured by the feature's value, red for high and blue for low.

In [ ]:
# GIVEN — run as-is
# Every SHAP value as a dot: position = the push on the prediction, colour = the feature's value
shap.summary_plot(shap_values, X_test)

### Questions

1. Which model wins on the test rows, and by how much over the tuned forest, in R² and in dollars of RMSE? Compare that with what tuning gained over Part 1's forest. Is the gap large enough to matter to someone pricing a house, and where would you spend the next hour of effort?
2. Compare XGBoost's SHAP ranking with Part 2's permutation ranking for the forest. Which features trade places, and what does that tell you about whose importance each ranking measures? Then read Latitude's `direction` and its row of dots in the chart: what do they say about XGBoost's predictions for block groups further north?

*Your answers here:*

1.
2.

---

## AI-Assisted Expansion: a SHAP summary, your prompt, checked by you

Part 3 took two cells to compute SHAP values and summarise them, and one more to chart them. A
function that does all of it for any fitted tree model is code you would reuse on every project,
and quick for AI to write. It also makes every choice your prompt leaves out, and several of those
choices change the numbers, some by a lot and some by less than one percent. No prompt is given:
you write it.

**The target.** Given Part 3's `xgb` and the test rows `X_test`, your function must reproduce
Part 3's SHAP summary: all 16 numbers, the `mean_abs_shap` and the `direction` of each of the eight
features, in the same order, from Latitude's **0.5241** and **−0.9371** to Population's **0.0238**
and **0.3491**.

**What the check calls.** `shap_summary(model, X)`: a fitted tree model, either an XGBoost model
or a scikit-learn random forest, and a DataFrame `X` of the rows to explain. It draws a chart of
the SHAP values and gives back a pandas DataFrame with one row per feature, largest
`mean_abs_shap` first, and three columns: `feature`, `mean_abs_shap` and `direction`. The check
runs it three times:
1. on a copy of Part 3's `xgb`, with `X_test`, beside Part 3's table;
2. on a smaller XGBoost model that the check fits, with every column renamed, explaining 1,000
   rows the lab never explained;
3. on a small random forest, fitted and explained the same way.

Runs 2 and 3 are compared with summaries the check computes as Part 3 did. A function that works
only with the lab's model, column names or rows fails there.

**What a good prompt states.** You can find each of these in Part 3's cells and notes.
- the function's name, its arguments in order, and what each one is;
- the models it must handle, and that each arrives already fitted;
- which explainer, and what data, if any, it is given besides the model;
- which rows are explained: all of `X`, or some of them;
- how each of the two summary numbers is computed, exactly;
- what to give back: the rows, their order, and the column names;
- the chart, and what it shows;
- what to leave out (example data, a model of its own, the lab's own names), and a comment on each
  line.

**What to do**
1. Write your round 1 prompt in the cell below and send it, exactly as written, to Claude
   ([claude.northeastern.edu](https://claude.northeastern.edu)) or ChatGPT, in a new chat.
2. Read the reply before you run anything. Paste only the function into the round 1 cell, and run
   the check below it.
3. If a line says `NO` or `missing`, or a run stops with an error, find the choice in the function
   that caused it, state that choice in a round 2 prompt, and paste and check the new function in
   the round 2 cells. A third round goes over round 2's function. Stop when every line says `yes`
   and you can explain each line, or when you decide to stop.
4. If everything says `yes` in round 1, skip rounds 2 and 3. Instead, in Question 1, name a choice
   the AI made that your prompt never stated, and write the line you would add to your prompt to
   state it.
5. Fill in the checking log and answer the three questions. Those and your prompts are graded,
   not the code.

**Reading the check.** Run 1 prints one line per number: Part 3's, the AI's, `same?`, and the %
apart, after a line that says whether the features come back in Part 3's order. Runs 2 and 3
print the order line and how many of their 16 numbers agree. Your numbers and the AI's come from
the same models in the same session. XGBoost's SHAP values are 32-bit numbers, and the same values
averaged in another order can differ in the sixth or seventh significant digit, so `same?` asks for
agreement to five significant digits. There is no other allowed gap: a line 0.1% apart says `NO`,
and it means the function did something different. The check does not say why a line differs;
you find that out by reading the function. It does not read the chart either: compare the AI's chart with
Part 3's yourself. When the order and every number agree in all three runs, the check ends with
`Passed`.

A correct function takes a second or two on run 1. If the check is still running after a minute,
the function is doing something Part 3 did not do.

Keep the function's name and argument order. If the AI also wrote example data or fitted a model
of its own, leave that out: it can replace the lab's.

In [ ]:
# GIVEN — run as-is (you need not follow this code; the cells below use it)
# The check, as a function: the cells below run it once for each round.
# It saves XGBoost, the test rows and Part 3's summary now, before anything is pasted.
import copy

saved_xgb, saved_X_test, saved_table = copy.deepcopy(xgb), X_test.copy(), shap_table.copy()
saved_pred = xgb.predict(X_test)
COLUMNS = ["mean_abs_shap", "direction"]

# Runs 2 and 3: every column renamed, two other models, and 1,000 training rows to explain
RENAMED = {"MedInc": "income", "HouseAge": "age", "AveRooms": "rooms", "AveBedrms": "bedrooms",
           "Population": "people", "AveOccup": "occupancy", "Latitude": "lat", "Longitude": "lon"}
renamed_train = X_train.rename(columns=RENAMED)
other_rows = renamed_train.iloc[:1000]
other_models = {
    "a smaller XGBoost": XGBRegressor(n_estimators=60, max_depth=3, learning_rate=0.2,
                                      random_state=7).fit(renamed_train, y_train),
    "a small random forest": RandomForestRegressor(n_estimators=30, max_depth=6,
                                                   random_state=7).fit(renamed_train, y_train),
}


def summary_wanted(model, rows):
    """Part 3's summary for another model and other rows: what runs 2 and 3 expect."""
    values = shap.TreeExplainer(model).shap_values(rows)
    table = pd.DataFrame({"mean_abs_shap": np.abs(values).mean(axis=0),
                          "direction": [np.corrcoef(rows[c], values[:, j])[0, 1]
                                        for j, c in enumerate(rows.columns)]},
                         index=rows.columns)
    return table.sort_values("mean_abs_shap", ascending=False)


wanted = {name: summary_wanted(model, other_rows) for name, model in other_models.items()}


def lab_data_unchanged():
    """True if xgb, X_test and Part 3's summary still hold what the lab made."""
    # try runs the lines under it; if they raise an error (say xgb is gone), except runs instead
    try:
        return (X_test.equals(saved_X_test) and shap_table.equals(saved_table)
                and np.array_equal(xgb.predict(saved_X_test), saved_pred))
    except Exception:
        return False


def agrees(want, ai):
    """Equal to five significant digits: 32-bit values averaged in another order still pass."""
    return bool(np.isclose(want, ai, rtol=1e-5, atol=0))


def show_pair(label, want, ai):
    """One line: the number wanted, the AI's, whether they agree, and how far apart they are."""
    pct = 100 * (ai - want) / abs(want)
    # sign and 2 significant digits, so a tiny gap prints like +9.6e-06 and 1.37 as +1.4;
    # from 100% on, a whole number with its sign, like -200
    apart = f"{pct:+.2g}" if abs(pct) < 100 else f"{pct:+.0f}"
    print(f"  {label:<26}{want:>10.4f}{ai:>10.4f}{'yes' if agrees(want, ai) else 'NO':>7}"
          f"{apart:>11}%")


def describe(value):
    """What the function gave back, in a few words."""
    if isinstance(value, (pd.Series, np.ndarray, list, tuple, dict)):
        return f"a {type(value).__name__} of length {len(value)}"
    return "None" if value is None else f"a {type(value).__name__}"


def as_table(result):
    """The DataFrame the function gave back, with lower-case column names and a feature column."""
    if isinstance(result, (tuple, list)):          # (chart, table) or (table, chart)
        tables = [r for r in result if isinstance(r, pd.DataFrame)]
        result = tables[0] if tables else result
    if not isinstance(result, pd.DataFrame):
        return None
    table = result.rename(columns=lambda c: str(c).strip().lower())
    if "feature" not in table.columns:             # the features may be the row labels
        table = table.rename_axis("feature").reset_index()
    return table


def ai_number(table, feature, column):
    """The AI's number for this feature and column, or None."""
    try:
        return float(table.loc[table["feature"].astype(str) == feature, column].iloc[0])
    except (IndexError, KeyError, TypeError, ValueError):
        return None


def one_run(fn, model, rows, want, every_line):
    """Run fn once and compare with want; True if the order and all 16 numbers agree."""
    try:
        # copies, so the function cannot change the lab's model or rows
        result = fn(copy.deepcopy(model), rows.copy())
    except Exception as err:
        print(f"  The AI's function stopped with an error. {type(err).__name__}: {err}")
        return False
    table = as_table(result)
    if table is None:
        print(f"  shap_summary gave back {describe(result)}. The check needs a pandas DataFrame.")
        return False
    if any(c not in table.columns for c in COLUMNS):
        print(f"  shap_summary gave back a DataFrame with columns {list(table.columns)}. "
              f"The check needs feature, {', '.join(COLUMNS)}.")
        return False
    ai_order = [str(f) for f in table["feature"]]
    same_order = ai_order == list(want.index)
    print(f"  order wanted : {', '.join(want.index)}")
    print(f"  the AI's rows: {', '.join(ai_order)}")
    print(f"  same features, same order? {'yes' if same_order else 'NO'}")
    agree = 0
    if every_line:
        print(f"\n  {'':<26}{'Part 3':>10}{'AI':>10}{'same?':>7}{'% apart':>12}")
    for feature in want.index:
        for column in COLUMNS:
            ai = ai_number(table, feature, column)
            agree += ai is not None and agrees(want.loc[feature, column], ai)
            if every_line and ai is None:
                print(f"  {feature + ', ' + column:<26}{want.loc[feature, column]:>10.4f}"
                      f"{'missing':>10}")
            elif every_line:
                show_pair(f"{feature}, {column}", want.loc[feature, column], ai)
    n = want.size
    if not every_line:
        print(f"  numbers the same: {agree} of {n}   {'yes' if agree == n else 'NO'}")
    return same_order and agree == n


def check_shap_summary(fn):
    """Run the AI's shap_summary on Part 3's model and rows, then on two others."""
    if not lab_data_unchanged():
        print("The pasted code changed xgb, X_test or Part 3's summary (shap_table). "
              "Paste only the function, then rerun the notebook from the top.")
        return
    # callable(x) is True when x is a function that can be called
    if not callable(fn):
        print("shap_summary should be a function. Paste the function the AI wrote.")
        return
    print("Run 1: Part 3's xgb, explaining X_test")
    passed = one_run(fn, saved_xgb, saved_X_test, saved_table, every_line=True)
    plt.show()                                 # run 1's chart, if the function drew one
    real_show = plt.show
    plt.show = lambda *args, **kwargs: None    # runs 2 and 3 draw no charts
    try:
        for run, (name, model) in enumerate(other_models.items(), start=2):
            print(f"\nRun {run}: {name}, columns renamed, explaining 1,000 other rows")
            passed = one_run(fn, model, other_rows, wanted[name], every_line=False) and passed
            plt.close("all")
    finally:
        plt.show = real_show
    if passed:
        print("\nPassed: the order and every number agree in all three runs.")
    else:
        print("\nIf a line says NO or missing, or a run stopped, find out why before you trust "
              "the function.")

**Your prompt, round 1.** Write it here, then send it exactly as written.

*Your prompt here:*

In [ ]:
# Paste the shap_summary function from round 1 here, then run the check below.
# Paste only the function: if the AI also wrote example data or fitted a model, leave it out.

In [ ]:
# CHECK — round 1
# To rerun this check, rerun the round 1 paste cell first: it checks whatever was pasted last
# globals() holds every name defined so far in this notebook
if "shap_summary" not in globals():
    print("Paste the AI's shap_summary function in the cell above first, keeping that name, "
          "and run that cell.")
else:
    shap_summary_r1 = shap_summary      # kept, so the round 2 check can tell the two apart
    check_shap_summary(shap_summary_r1)

**Your prompt, round 2** (and round 3 under it, if you need one). Keep what worked in round 1,
and state what the AI had to choose on its own.

*Your prompt here:*

In [ ]:
# Paste the shap_summary function from round 2 here, then run the check below.
# Paste only the function: if the AI also wrote example data or fitted a model, leave it out.

In [ ]:
# CHECK — round 2 (and round 3)
# "is" asks whether two names hold the very same function: if so, nothing new was pasted
if "shap_summary" not in globals() or shap_summary is globals().get("shap_summary_r1"):
    print("Paste the AI's round 2 shap_summary function in the cell above first, "
          "keeping that name, and run that cell.")
else:
    check_shap_summary(shap_summary)

### Checking log

This is graded, with your prompts and the three answers below. One row per round (a third round, if
any, goes in round 2's row): what you checked, how you checked it, and what you found. A row that
names the lines that differed, how far apart they were, and the line of the AI's code responsible
earns credit; "it worked" does not. The chart row is the AI's chart beside Part 3's, which the
check does not read. The last row is where you stopped, and why.

| Round | What I checked | How | What I found |
|---|---|---|---|
| 1 |  |  |  |
| 2 (write "not needed" if round 1 was all yes) |  |  |  |
| Chart |  |  |  |
| Where I stopped |  |  |  |

### Questions

1. **Critique, in your own words.** Name one choice the AI made that your prompt never stated, quote the line where it makes it, and say whether you would keep it and why. A choice that caused a `NO`, a `missing` or an error counts. If everything said `yes` in round 1, also write the prompt line that would state that choice.
2. **Ownership.** Quote the lines of your final function that compute `mean_abs_shap` and `direction`, and explain in 2-3 sentences, in your own words, what they do and why they give Part 3's numbers.
3. **Judgment: where you stopped.** Which round did you stop at, and why do you, or do you not, trust that function on a model and data the check has never seen? Then: when would you ask AI to write code like this, and when would you write it yourself?

*Your answers here:*

1.
2.
3.

---

## Extension (Optional — 15 min): One Prediction at a Time

Part 3 summarised SHAP values over all the test rows. A SHAP value is defined one prediction at a
time, and a waterfall chart shows one: it starts at the base value and adds one feature's SHAP value
at a time until it reaches the prediction.

**The task.** In the cell below, and in new cells under it, pick three test rows: the one XGBoost predicts highest, the one it predicts lowest, and the one
   it misses by most (the largest absolute gap between its prediction and `y_test`). `np.argmax`
   and `np.argmin` give the position of the largest and the smallest entry of an array. Draw a
   waterfall chart for each, where `i` is the row's position:
   `shap.plots.waterfall(shap.Explanation(values=shap_values[i], base_values=base_value,
   data=X_test.iloc[i].values, feature_names=list(X_test.columns)))`

If you skip the Extension, leave its cell unrun, so your submitted notebook shows no error.

In [ ]:
# YOUR TASK — optional: three waterfall charts

### Interpretation Questions

1. For the row XGBoost misses by most, which features pushed its prediction away from the true value? Is anything about that block group unusual?
2. In the rows with the highest and the lowest prediction, which feature pushes hardest? Is it the feature at the top of Part 3's `mean_abs_shap` ranking, and how can one row's biggest push differ from the largest average?

*Your answers here:*

1.

2.

---
## Digital Portfolio: your README

Copy the prompt below into Claude or ChatGPT. Replace every `[YOUR VALUE]` with the number from
your own output first. **Do NOT ask the AI to write Python code — only documentation.**

```
I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Found a bug in a model comparison on California Housing:
  [ONE LINE: WHAT WAS WRONG]. The analyst reported R² [YOUR VALUE] for
  the random forest; corrected, it is [YOUR VALUE], against [YOUR VALUE]
  for a single tree and [YOUR VALUE] for Ridge
* Explained why a feature-importance ranking cannot support a policy
  recommendation, and compared impurity (MDI) with permutation importance
* Tuned the forest by cross-validation and compared it with XGBoost on the
  test rows: test R² [YOUR VALUE] for the tuned forest, [YOUR VALUE] for XGBoost
* Read XGBoost's SHAP values: the feature that moved its predictions most
  was [YOUR VALUE]
* Wrote my own prompt asking an AI for a function that summarises SHAP
  values, went [YOUR VALUE] rounds, and checked its 16 numbers against my
  own: [YOUR RESULT]

**Please write a README.md entry including:**
1. Project Title: Tree-Based Models: Random Forests, XGBoost and SHAP
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'comprehensive' or
'cutting-edge'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ5200-lab19-random-forests`

**First time?** Read the **GitHub Setup and Repository Guide** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ5200-lab19-random-forests`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ5200-lab19-random-forests`, branch `main`,
   commit message `Lab 19 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the README prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.